In [11]:
!pip install langchain-schema

ERROR: Could not find a version that satisfies the requirement langchain-schema (from versions: none)
ERROR: No matching distribution found for langchain-schema


In [13]:
import os
import glob

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document

# import gradio as gr

load_dotenv(override=True)

False

In [14]:
print("Environment variables loaded.")

Environment variables loaded.


In [15]:
import json

with open("series_2843071_raw.json", "r", encoding="utf-8") as f:
    raw = json.load(f)

series = raw["seriesState"]

extracted = {
    "series_id": "2843071",
    "format": series.get("format"),
    "maps": []
}

for game in series.get("games", []):
    map_name = game["map"]["name"]

    map_entry = {
        "map": map_name,
        "teams": []
    }

    # Look at rounds → teams → players
    for segment in game.get("segments", []):
        if segment["type"] != "round":
            continue

        for team in segment.get("teams", []):
            team_name = team.get("name")
            side = team.get("side")
            won = team.get("won")

            players = []
            for p in team.get("players", []):
                players.append({
                    "player": p.get("name"),
                    "kills": p.get("kills"),
                    "deaths": p.get("deaths"),
                    "headshots": p.get("headshots")
                })

            map_entry["teams"].append({
                "team": team_name,
                "side": side,
                "won_round": won,
                "players": players
            })

    extracted["maps"].append(map_entry)


In [16]:
documents = []

for m in extracted["maps"]:
    text = f"Map: {m['map']}\n"

    for t in m["teams"]:
        text += f"\nTeam {t['team']} played as {t['side']}.\n"
        text += "Players:\n"
        for p in t["players"]:
            text += f"- {p['player']}: {p['kills']} kills, {p['deaths']} deaths\n"

    doc = Document(
        page_content=text,
        metadata={
            "series_id": extracted["series_id"],
            "map": m["map"],
            "type": "map_scouting"
        }
    )

    documents.append(doc)

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 400, chunk_overlap = 40)
chunks= text_splitter.split_documents(documents) 

In [ ]:
# embeddings = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
embeddings = HuggingFaceEmbeddings(model_name = "BAAI/bge-large-en-v1.5")

db_name = "vector_db"

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()
    
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)